In [5]:
import tensorflow as tf
from keras import layers, Model
import numpy as np

latent_dim = 2 

class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs 
        return mean + tf.exp(0.5 * log_var) * tf.random.normal(tf.shape(mean))
    
encoder = tf.keras.Sequential([
    layers.InputLayer((28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(latent_dim * 2)
])

decoder = tf.keras.Sequential([
    layers.InputLayer((latent_dim, )),
    layers.Dense(128, activation='relu'),
    layers.Dense(28*28, activation='sigmoid'),
    layers.Reshape((28, 28, 1))
])

class VAE(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder 
        self.decoder = decoder 
        self.sampling = Sampling()

    def call(self, x):
        z_mean, z_log_var = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        z = self.sampling((z_mean, z_log_var))
        self.add_loss(-0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)))
        return self.decoder(z)
    
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255 
x_train = np.expand_dims(x_train, -1)

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam', loss=tf.keras.losses.BinaryCrossentropy())
vae.fit(x_train, x_train, epochs=3, batch_size=64)

Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2943
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2634
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.2631


In [13]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
import numpy as np

latent_dim = 100

generator = Sequential([
    layers.Dense(7*7*128, input_shape=(latent_dim,), activation='relu'),
    layers.Reshape((7, 7, 128)),
    layers.Conv2DTranspose(64, 4, 2, 'same', activation='relu'),
    layers.Conv2DTranspose(1, 4, 2, 'same', activation='sigmoid')
])

discriminator = Sequential([
    layers.InputLayer((28,28,1)),
    layers.Conv2D(64, 4, 2, 'same', activation='relu'),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
])
discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
discriminator.trainable = False

gan_input = layers.Input(shape=(latent_dim,))
gan = tf.keras.Model(gan_input, discriminator(generator(gan_input)))
gan.compile(optimizer='adam', loss='binary_crossentropy')

(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_train = np.expand_dims(x_train, -1)

batch_size = 64
steps = 3 * (x_train.shape[0] // batch_size)  # approx 3 epochs

for step in range(steps):
    noise = np.random.normal(size=(batch_size, latent_dim))
    gen_imgs = generator.predict(noise, verbose=0)
    real_imgs = x_train[np.random.randint(0, x_train.shape[0], batch_size)]

    images = np.vstack([real_imgs, gen_imgs])
    labels = np.vstack([np.ones((batch_size,1)), np.zeros((batch_size,1))])
    d_loss = discriminator.train_on_batch(images, labels)

    noise = np.random.normal(size=(batch_size, latent_dim))
    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if step % (x_train.shape[0] // batch_size) == 0:
        epoch = step // (x_train.shape[0] // batch_size)
        print(f"Epoch {epoch}: D loss={d_loss[0]:.4f}, acc={d_loss[1]:.4f}, G loss={g_loss:.4f}")


/Users/orbinsunny/.pyenv/versions/3.12.3/lib/python3.12/site-packages/keras/src/backend/tensorflow/trainer.py:83: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Epoch 0: D loss=0.7236, acc=0.2969, G loss=0.6300
Epoch 1: D loss=1.2856, acc=0.3027, G loss=0.1788
Epoch 2: D loss=1.3347, acc=0.3019, G loss=0.1561
